# 00 — Download the dataset (optional)

Downloads the whole improved CICIDS2017 + CSE-CIC-IDS2018 dataset as one zip (~11 GB, 37 GB unzipped) from Kaggle ([ernie55ernie/improved-cicids2017-and-csecicids2018](https://www.kaggle.com/datasets/ernie55ernie/improved-cicids2017-and-csecicids2018)), then unzips it to where `eda` and `02_ensemble_models_cicids2017` read from. No Kaggle login is needed.

**Nothing downloads by default.** Set `DOWNLOAD = True` to run it. An existing zip is reused, and an interrupted download resumes.

In [ ]:
DOWNLOAD = False   # set True to download the ~11 GB zip
UNZIP = True       # extract into ROOT after downloading

ROOT = './Data/'   # folder the other notebooks read
ZIP = ROOT + 'archive.zip'

In [ ]:
import os, zipfile, requests

URL = 'https://www.kaggle.com/api/v1/datasets/download/ernie55ernie/improved-cicids2017-and-csecicids2018'

if not DOWNLOAD:
    print('DOWNLOAD is False; nothing to do.')
elif os.path.exists(ZIP):
    print(f'Zip already present: {ZIP}')
else:
    part = ZIP + '.part'
    done = os.path.getsize(part) if os.path.exists(part) else 0
    with requests.get(URL, headers={'Range': f'bytes={done}-'}, stream=True, timeout=60) as r:
        if r.status_code != 416:          # 416 = .part is already complete
            r.raise_for_status()
            if r.status_code != 206:      # server ignored the resume request
                done = 0
            total = done + int(r.headers.get('Content-Length', 0))
            with open(part, 'ab' if done else 'wb') as f:
                for i, chunk in enumerate(r.iter_content(chunk_size=8 << 20)):
                    f.write(chunk)
                    done += len(chunk)
                    if i % 64 == 0:
                        print(f'\r{done/1e9:.1f} / {total/1e9:.1f} GB', end='', flush=True)
            print()
    os.replace(part, ZIP)
    print(f'Saved {ZIP}')

In [ ]:
if DOWNLOAD and UNZIP:
    with zipfile.ZipFile(ZIP) as z:
        todo = [m for m in z.infolist()
                if not (os.path.exists(os.path.join(ROOT, m.filename))
                        and os.path.getsize(os.path.join(ROOT, m.filename)) == m.file_size)]
        print(f'Extracting {len(todo)} of {len(z.infolist())} files into {ROOT}')
        for m in todo:
            z.extract(m, ROOT)
            print('  ', m.filename)